# Data Processing and Consolidation for US and Germany Energy Data

This notebook processes and consolidates energy data for various regions in the United States and Germany, spanning the years 2021, 2022, and 2023. The processed data is then saved into Parquet files for further analysis. The key steps and functionalities of the notebook are as follows:

## Key Steps and Functionalities

1. **Define Paths and Region Mappings:**
   - Paths to raw data directories for the US and Germany are specified.
   - A mapping for US regions is defined to categorize data files based on their prefixes.

2. **Initialize DataFrames:**
   - Empty DataFrames are initialized to store daily and hourly data for the years 2021, 2022, and 2023.

3. **Helper Functions:**
   - `move_columns_to_front(df, columns)`: Reorders columns in the DataFrame to move specified columns to the front.
   - `drop_nan_rows(df, numeric_columns)`: Drops rows with NaN values in specified numeric columns.

4. **Process Region Data:**
   - The `process_region_data` function processes and appends regional data to the initialized DataFrames. It handles data concatenation, year extraction, and resampling for daily and hourly aggregations.

5. **Iterate Over US Regions:**
   - The notebook iterates over the region mapping for the US, reads corresponding CSV files, and processes the data using the `process_region_data` function.

6. **Process German Data:**
   - Similar processing is done for Germany. The notebook reads CSV files for Germany and processes the data for daily and hourly aggregations.

7. **Save Processed Data:**
   - The processed DataFrames for 2021, 2022, and 2023 (both daily and hourly) are saved into Parquet files in the specified `cleaned_data_path`.

## Data Processing Details

### US Region Data Processing
- **Concatenation:** All DataFrames for a region are concatenated.
- **Year Extraction:** The year is extracted from the 'Datetime (UTC)' column.
- **Yearly Processing:**
  - **2021 and 2022:** Data is resampled to daily frequency, columns are reordered, NaN rows are dropped, and the data is appended to the respective DataFrames.
  - **2023:** Both hourly and daily data are processed. Hourly data is grouped by hour, and daily data is resampled to daily frequency. Columns are reordered, NaN rows are dropped, and the data is appended to hourly and daily DataFrames for 2023.

### German Data Processing
- **Year Extraction:** Similar to US data, the year is extracted from the 'Datetime (UTC)' column.
- **Yearly Processing:** Follows the same procedure as US data for daily and hourly aggregations.

## Output
- The processed data is saved into the following Parquet files:
  - `all_regions_2021_daily.parquet`
  - `all_regions_2022_daily.parquet`
  - `all_regions_2023_daily.parquet`
  - `all_regions_2023_hourly.parquet`

By the end of this notebook, all the raw data is efficiently processed, aggregated, and saved into consolidated Parquet files, ready for further analysis.

In [42]:
import pandas as pd
import os

In [43]:
# Define the path to the raw data
data_path = "raw_data/us/"
germany_data_path = "raw_data/de/"
cleaned_data_path = "../results/energy_data/cleaned_data/"

In [44]:
# Define the region mappings
region_mapping = {
    'US-West': ['US-NW-', 'US-SW-', 'US-CAL-'],
    'US-Central': ['US-CENT-', 'US-MIDW-', 'US-TEX-', 'US-TEN-'],
    'US-East': ['US-CAR-', 'US-MIDA-', 'US-NE-', 'US-NY-', 'US-SE-', 'US-FLA-'],
    'US-Average': ['US_AVG_']
}

# Numeric columns that are of interest
numeric_columns = [
    "Carbon Intensity gCO₂eq/kWh (direct)",
    "Carbon Intensity gCO₂eq/kWh (LCA)",
    "Low Carbon Percentage",
    "Renewable Percentage"
]

In [45]:
# Initialize empty dataframes for each year and for 2023 (hourly and daily)
df_2021_daily = pd.DataFrame()
df_2022_daily = pd.DataFrame()
df_2023_hourly = pd.DataFrame()
df_2023_daily = pd.DataFrame()

In [46]:
# Helper function to move specified columns to the front
def move_columns_to_front(df, columns_to_move):
    columns = columns_to_move + [col for col in df.columns if col not in columns_to_move]
    return df[columns]

In [47]:
# Helper function to drop rows where all numeric columns have NaN values
def drop_nan_rows(df, numeric_columns):
    return df.dropna(axis=0, how='any')

### Description of the `process_region_data` Function

The `process_region_data` function processes and appends regional data for the years 2021, 2022, and 2023.

1. **Inputs:**
   - `region`: Name of the region.
   - `region_data`: List of DataFrames with regional data.

2. **Global DataFrames:**
   - `df_2021_daily`, `df_2022_daily`: Store daily data for 2021 and 2022.
   - `df_2023_hourly`, `df_2023_daily`: Store hourly and daily data for 2023.

3. **Processing Steps:**
   - Concatenates `region_data` into `all_data_for_region`.
   - Extracts the year from 'Datetime (UTC)' and adds it as `Year`.
   - Processes data by year:
     - **2021, 2022:** Resamples to daily frequency, adds 'Country' and 'Zone', reorders columns, drops NaNs, appends to `df_2021_daily` and `df_2022_daily`.
     - **2023:** 
       - **Hourly:** Groups by hour, adds 'Country' and 'Zone', reorders columns, drops NaNs, appends to `df_2023_hourly`.
       - **Daily:** Resamples to daily frequency, adds 'Country' and 'Zone', reorders columns, drops NaNs, appends to `df_2023_daily`.

### Helper Functions:
- `move_columns_to_front(df, columns)`: Reorders columns.
- `drop_nan_rows(df, numeric_columns)`: Drops rows with NaNs.

This function organizes and appends regional data into yearly DataFrames, ensuring correct daily and hourly aggregations.

In [48]:
# Function to process and append the data
def process_region_data(region, region_data):
    """Processes the region data, appends to the corresponding year dataframes."""
    global df_2021_daily, df_2022_daily, df_2023_hourly, df_2023_daily

    # Concatenate all data for the region
    all_data_for_region = pd.concat(region_data, ignore_index=True)

    # Extract year from the 'Datetime (UTC)' column
    all_data_for_region['Year'] = all_data_for_region['Datetime (UTC)'].dt.year.astype(int)

    # Separate and append data for each year
    for year in [2021, 2022, 2023]:
        data_for_year = all_data_for_region[all_data_for_region['Year'] == year].copy()

        if not data_for_year.empty:
            if year == 2023:
                # Group hourly data for 2023 and append
                hourly_data_2023 = data_for_year.groupby([pd.Grouper(key='Datetime (UTC)', freq='h')]).mean().reset_index()
                hourly_data_2023['Country'] = 'USA'
                hourly_data_2023['Zone'] = region
                hourly_data_2023 = move_columns_to_front(hourly_data_2023, ['Country', 'Zone', 'Datetime (UTC)'])
                hourly_data_2023 = drop_nan_rows(hourly_data_2023, numeric_columns)
                df_2023_hourly = pd.concat([df_2023_hourly, hourly_data_2023], ignore_index=True)

                # Aggregate to daily data for 2023
                daily_data_2023 = data_for_year.resample('D', on='Datetime (UTC)').mean().reset_index()
                daily_data_2023['Country'] = 'USA'
                daily_data_2023['Zone'] = region
                daily_data_2023 = move_columns_to_front(daily_data_2023, ['Country', 'Zone', 'Datetime (UTC)'])
                daily_data_2023 = drop_nan_rows(daily_data_2023, numeric_columns)
                df_2023_daily = pd.concat([df_2023_daily, daily_data_2023], ignore_index=True)
            else:
                # Append daily data for 2021 and 2022
                daily_data = data_for_year.resample('D', on='Datetime (UTC)').mean().reset_index()
                daily_data['Country'] = 'USA'
                daily_data['Zone'] = region
                daily_data = move_columns_to_front(daily_data, ['Country', 'Zone', 'Datetime (UTC)'])
                daily_data = drop_nan_rows(daily_data, numeric_columns)

                if year == 2021:
                    df_2021_daily = pd.concat([df_2021_daily, daily_data], ignore_index=True)
                elif year == 2022:
                    df_2022_daily = pd.concat([df_2022_daily, daily_data], ignore_index=True)


In [49]:
# Iterate over the region mapping for the US
for region, prefixes in region_mapping.items():
    # List to store dataframes for the current region
    region_data = []

    # Iterate over all files in the raw data directory
    for file_name in os.listdir(data_path):
        # Check if the file name starts with any of the region-specific prefixes
        if any(file_name.startswith(prefix) for prefix in prefixes):
            # Read the CSV file
            df = pd.read_csv(os.path.join(data_path, file_name), parse_dates=['Datetime (UTC)'])
            # Filter for the columns of interest
            df = df[["Datetime (UTC)"] + numeric_columns]
            region_data.append(df)

    # Process the region's data and append to the consolidated dataframes
    if region_data:
        process_region_data(region, region_data)

In [50]:
# Process the German data similarly
germany_files = [
    'DE_2021_daily.csv',
    'DE_2022_daily.csv',
    #'DE_2023_daily.csv',
    'DE_2023_hourly.csv'
]

This cell processes CSV files containing data for Germany, extracting and aggregating data by year (2021, 2022, 2023). Specifically:

- Reads each CSV file and extracts the year from the 'Datetime (UTC)' column.
- For 2023:
  - Aggregates data hourly and daily, adding 'Country', 'Zone', and 'Year' columns.
  - Appends the processed data to `df_2023_hourly` and `df_2023_daily` respectively.
- For 2021 and 2022:
  - Aggregates data daily, adding 'Country', 'Zone', and 'Year' columns.
  - Appends the processed data to `df_2021_daily` and `df_2022_daily`.

Helper functions `move_columns_to_front` and `drop_nan_rows` are used for column ordering and removing rows with NaNs in numeric columns.

In [51]:
for file_name in germany_files:
    # Read the CSV file
    df = pd.read_csv(os.path.join(germany_data_path, file_name), parse_dates=['Datetime (UTC)'])

    # Extract year from the 'Datetime (UTC)' column
    df['Year'] = df['Datetime (UTC)'].dt.year.astype(int)

    # Process data for Germany
    for year in [2021, 2022, 2023]:
        data_for_year = df[df['Year'] == year].copy()

        if not data_for_year.empty:
            if year == 2023:
                # Group hourly data for 2023 and append (using only numeric columns)
                hourly_data_2023 = data_for_year.groupby([pd.Grouper(key='Datetime (UTC)', freq='h')])[numeric_columns].mean().reset_index()
                
                # Add non-numeric columns after aggregation
                hourly_data_2023['Country'] = 'Germany'
                hourly_data_2023['Zone'] = 'Germany'
                hourly_data_2023['Year'] = year
                hourly_data_2023 = move_columns_to_front(hourly_data_2023, ['Country', 'Zone', 'Datetime (UTC)'])
                hourly_data_2023 = drop_nan_rows(hourly_data_2023, numeric_columns)
                df_2023_hourly = pd.concat([df_2023_hourly, hourly_data_2023], ignore_index=True)

                # Aggregate to daily data for 2023 (using only numeric columns)
                daily_data_2023 = data_for_year.resample('D', on='Datetime (UTC)')[numeric_columns].mean().reset_index()
                
                # Add non-numeric columns after aggregation
                daily_data_2023['Country'] = 'Germany'
                daily_data_2023['Zone'] = 'Germany'
                daily_data_2023['Year'] = year
                daily_data_2023 = move_columns_to_front(daily_data_2023, ['Country', 'Zone', 'Datetime (UTC)'])
                daily_data_2023 = drop_nan_rows(daily_data_2023, numeric_columns)
                df_2023_daily = pd.concat([df_2023_daily, daily_data_2023], ignore_index=True)
            else:
                # Append daily data for 2021 and 2022 (using only numeric columns)
                daily_data = data_for_year.resample('D', on='Datetime (UTC)')[numeric_columns].mean().reset_index()
                
                # Add non-numeric columns after aggregation
                daily_data['Country'] = 'Germany'
                daily_data['Zone'] = 'Germany'
                daily_data['Year'] = year
                daily_data = move_columns_to_front(daily_data, ['Country', 'Zone', 'Datetime (UTC)'])
                daily_data = drop_nan_rows(daily_data, numeric_columns)

                if year == 2021:
                    df_2021_daily = pd.concat([df_2021_daily, daily_data], ignore_index=True)
                elif year == 2022:
                    df_2022_daily = pd.concat([df_2022_daily, daily_data], ignore_index=True)

In [52]:
df_2023_hourly

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2023-01-01 00:00:00,315.200690,367.744828,52.148621,46.793448,2023.0
1,USA,US-West,2023-01-01 01:00:00,313.973793,366.243103,52.407241,47.267241,2023.0
2,USA,US-West,2023-01-01 02:00:00,315.176207,367.120690,52.518966,47.355517,2023.0
3,USA,US-West,2023-01-01 03:00:00,316.315517,368.106207,52.432414,47.097931,2023.0
4,USA,US-West,2023-01-01 04:00:00,319.809310,371.676207,51.885172,46.285517,2023.0
...,...,...,...,...,...,...,...,...
43109,Germany,Germany,2023-12-31 19:00:00,138.950000,186.760000,84.330000,80.710000,2023.0
43110,Germany,Germany,2023-12-31 20:00:00,137.630000,186.060000,84.130000,80.630000,2023.0
43111,Germany,Germany,2023-12-31 21:00:00,139.160000,188.340000,83.940000,81.590000,2023.0
43112,Germany,Germany,2023-12-31 22:00:00,139.130000,188.800000,83.830000,81.970000,2023.0


In [53]:
df_2023_daily

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2023-01-01,331.906897,383.615287,50.917744,44.584899,2023.0
1,USA,US-West,2023-01-02,342.101796,396.722557,48.321394,42.457888,2023.0
2,USA,US-West,2023-01-03,329.307744,383.656092,49.683606,43.662529,2023.0
3,USA,US-West,2023-01-04,303.097931,354.633420,53.345489,47.081897,2023.0
4,USA,US-West,2023-01-05,302.592069,354.773836,53.055848,46.833103,2023.0
...,...,...,...,...,...,...,...,...
1817,Germany,Germany,2023-12-27,229.412083,289.484583,73.339583,67.439167,2023.0
1818,Germany,Germany,2023-12-28,138.803750,183.870833,84.619167,83.346667,2023.0
1819,Germany,Germany,2023-12-29,131.847500,175.907917,85.420000,84.338333,2023.0
1820,Germany,Germany,2023-12-30,164.925000,215.579167,82.023333,78.514167,2023.0


In [54]:
df_2021_daily

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2021-01-01,320.708966,369.515862,53.025862,46.868621,2021.0
1,USA,US-West,2021-01-02,304.941379,353.307931,55.138966,49.175517,2021.0
2,USA,US-West,2021-01-03,287.977241,335.264483,57.533103,51.532414,2021.0
3,USA,US-West,2021-01-04,301.457931,349.400690,55.708966,49.951034,2021.0
4,USA,US-West,2021-01-05,291.010000,337.753793,57.602069,51.740690,2021.0
...,...,...,...,...,...,...,...,...
1773,Germany,Germany,2021-12-27,406.270000,469.150000,58.230000,44.320000,2021.0
1774,Germany,Germany,2021-12-28,360.400000,418.580000,63.050000,49.160000,2021.0
1775,Germany,Germany,2021-12-29,433.230000,497.740000,56.090000,41.080000,2021.0
1776,Germany,Germany,2021-12-30,191.880000,238.830000,79.320000,67.490000,2021.0


In [55]:
df_2022_daily

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2022-01-01,313.493793,357.028276,56.234828,50.030690,2022.0
1,USA,US-West,2022-01-02,309.970690,359.255517,55.632759,49.719310,2022.0
2,USA,US-West,2022-01-03,311.830345,360.850000,55.339310,49.482069,2022.0
3,USA,US-West,2022-01-04,288.826552,336.274828,58.160345,52.016207,2022.0
4,USA,US-West,2022-01-05,307.669310,355.478966,55.346207,48.990690,2022.0
...,...,...,...,...,...,...,...,...
1791,Germany,Germany,2022-12-27,250.280000,302.190000,75.590000,68.880000,2022.0
1792,Germany,Germany,2022-12-28,178.210000,224.230000,82.010000,74.780000,2022.0
1793,Germany,Germany,2022-12-29,194.060000,240.940000,80.940000,74.920000,2022.0
1794,Germany,Germany,2022-12-30,184.580000,232.480000,81.390000,74.200000,2022.0


## Adding further contires

Further countries have to be located in the new folder inside the raw_data folder. 
We iterate over all the subfolders in the new folder to extract all the data. 

In [56]:
new_data_path = "raw_data/new/"

# Define region mappings
region_mappings = {
    'ch': {'Country': 'Switzerland', 'Zone': 'Switzerland'},
    'fr': {'Country': 'France', 'Zone': 'France'},
    'is': {'Country': 'Iceland', 'Zone': 'Iceland'},
    'no': {'Country': 'Norway', 'Zone': 'Norway'},
    'se': {'Country': 'Sweden', 'Zone': 'Sweden'}
}


In [57]:
# Iterate over each region folder in the "new" directory
for region_code, region_info in region_mappings.items():
    region_data_path = os.path.join(new_data_path, region_code)
    
    # List all files in the region directory
    region_files = [file for file in os.listdir(region_data_path) if file.endswith('.csv')]

    # Iterate through each file in the region folder
    for file_name in region_files:
        # Read the CSV file
        df = pd.read_csv(os.path.join(region_data_path, file_name), parse_dates=['Datetime (UTC)'])

        # Extract year from the 'Datetime (UTC)' column
        df['Year'] = df['Datetime (UTC)'].dt.year.astype(int)

        # Process data for each year (2021, 2022, 2023)
        for year in [2021, 2022, 2023]:
            data_for_year = df[df['Year'] == year].copy()

            if not data_for_year.empty:
                if year == 2023:
                    # Group hourly data for 2023
                    hourly_data_2023 = data_for_year.groupby([pd.Grouper(key='Datetime (UTC)', freq='h')])[numeric_columns].mean().reset_index()
                    
                    # Add non-numeric columns after aggregation
                    hourly_data_2023['Country'] = region_info['Country']
                    hourly_data_2023['Zone'] = region_info['Zone']
                    hourly_data_2023['Year'] = year
                    hourly_data_2023 = move_columns_to_front(hourly_data_2023, ['Country', 'Zone', 'Datetime (UTC)'])
                    hourly_data_2023 = drop_nan_rows(hourly_data_2023, numeric_columns)
                    df_2023_hourly = pd.concat([df_2023_hourly, hourly_data_2023], ignore_index=True)

                    # Aggregate to daily data for 2023
                    daily_data_2023 = data_for_year.resample('D', on='Datetime (UTC)')[numeric_columns].mean().reset_index()

                    # Add non-numeric columns after aggregation
                    daily_data_2023['Country'] = region_info['Country']
                    daily_data_2023['Zone'] = region_info['Zone']
                    daily_data_2023['Year'] = year
                    daily_data_2023 = move_columns_to_front(daily_data_2023, ['Country', 'Zone', 'Datetime (UTC)'])
                    daily_data_2023 = drop_nan_rows(daily_data_2023, numeric_columns)
                    df_2023_daily = pd.concat([df_2023_daily, daily_data_2023], ignore_index=True)
                else:
                    # Append daily data for 2021 and 2022
                    daily_data = data_for_year.resample('D', on='Datetime (UTC)')[numeric_columns].mean().reset_index()

                    # Add non-numeric columns after aggregation
                    daily_data['Country'] = region_info['Country']
                    daily_data['Zone'] = region_info['Zone']
                    daily_data['Year'] = year
                    daily_data = move_columns_to_front(daily_data, ['Country', 'Zone', 'Datetime (UTC)'])
                    daily_data = drop_nan_rows(daily_data, numeric_columns)

                    if year == 2021:
                        df_2021_daily = pd.concat([df_2021_daily, daily_data], ignore_index=True)
                    elif year == 2022:
                        df_2022_daily = pd.concat([df_2022_daily, daily_data], ignore_index=True)

In [58]:
df_2023_hourly

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2023-01-01 00:00:00,315.200690,367.744828,52.148621,46.793448,2023.0
1,USA,US-West,2023-01-01 01:00:00,313.973793,366.243103,52.407241,47.267241,2023.0
2,USA,US-West,2023-01-01 02:00:00,315.176207,367.120690,52.518966,47.355517,2023.0
3,USA,US-West,2023-01-01 03:00:00,316.315517,368.106207,52.432414,47.097931,2023.0
4,USA,US-West,2023-01-01 04:00:00,319.809310,371.676207,51.885172,46.285517,2023.0
...,...,...,...,...,...,...,...,...
86767,Sweden,Sweden,2023-12-31 19:00:00,22.420000,46.730000,97.490000,69.710000,2023.0
86768,Sweden,Sweden,2023-12-31 20:00:00,30.280000,55.300000,96.700000,67.590000,2023.0
86769,Sweden,Sweden,2023-12-31 21:00:00,31.920000,57.070000,96.500000,66.940000,2023.0
86770,Sweden,Sweden,2023-12-31 22:00:00,30.650000,55.950000,96.600000,66.630000,2023.0


In [59]:
df_2023_daily

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2023-01-01,331.906897,383.615287,50.917744,44.584899,2023.0
1,USA,US-West,2023-01-02,342.101796,396.722557,48.321394,42.457888,2023.0
2,USA,US-West,2023-01-03,329.307744,383.656092,49.683606,43.662529,2023.0
3,USA,US-West,2023-01-04,303.097931,354.633420,53.345489,47.081897,2023.0
4,USA,US-West,2023-01-05,302.592069,354.773836,53.055848,46.833103,2023.0
...,...,...,...,...,...,...,...,...
3640,Sweden,Sweden,2023-12-27,5.374167,25.442083,99.411250,68.737500,2023.0
3641,Sweden,Sweden,2023-12-28,15.717917,39.151667,98.210417,66.931667,2023.0
3642,Sweden,Sweden,2023-12-29,23.400417,48.042083,97.499583,64.220833,2023.0
3643,Sweden,Sweden,2023-12-30,19.538750,43.592083,97.849167,65.999167,2023.0


In [60]:
df_2022_daily

,Country,Zone,Datetime (UTC),Carbon Intensity gCO₂eq/kWh (direct),Carbon Intensity gCO₂eq/kWh (LCA),Low Carbon Percentage,Renewable Percentage,Year
0,USA,US-West,2022-01-01,313.493793,357.028276,56.234828,50.030690,2022.0
1,USA,US-West,2022-01-02,309.970690,359.255517,55.632759,49.719310,2022.0
2,USA,US-West,2022-01-03,311.830345,360.850000,55.339310,49.482069,2022.0
3,USA,US-West,2022-01-04,288.826552,336.274828,58.160345,52.016207,2022.0
4,USA,US-West,2022-01-05,307.669310,355.478966,55.346207,48.990690,2022.0
...,...,...,...,...,...,...,...,...
3616,Sweden,Sweden,2022-12-27,11.020000,35.020000,98.730000,72.050000,2022.0
3617,Sweden,Sweden,2022-12-28,14.610000,41.050000,98.310000,71.210000,2022.0
3618,Sweden,Sweden,2022-12-29,10.690000,34.440000,98.810000,70.820000,2022.0
3619,Sweden,Sweden,2022-12-30,10.610000,34.190000,98.750000,68.160000,2022.0


In [61]:
df_2021_daily.to_parquet(os.path.join(cleaned_data_path, 'all_regions_2021_daily.parquet')) 
df_2022_daily.to_parquet(os.path.join(cleaned_data_path, 'all_regions_2022_daily.parquet')) 
df_2023_daily.to_parquet(os.path.join(cleaned_data_path, 'all_regions_2023_daily.parquet')) 
df_2023_hourly.to_parquet(os.path.join(cleaned_data_path, 'all_regions_2023_hourly.parquet'))

print("All data has been processed and saved into consolidated Parquet files.")

All data has been processed and saved into consolidated Parquet files.
